# 02 — Uncertainty Head Training

Trains the evidential (Normal-Inverse-Gamma) uncertainty head on synthetic
PyBullet sequences (exact ground truth, zero download) plus, optionally, a
small ClearGrasp real-image subset. This is the core novel component of
UGAP -- separating aleatoric ("genuinely ambiguous surface") from
epistemic ("model hasn't seen enough views") uncertainty, then checking
that epistemic uncertainty actually tracks reconstruction error better
than DUSt3R's native confidence does.

Run `00_setup_colab.ipynb` first.


In [ ]:
import sys
sys.path.append('.')
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

from ugap.reconstruction import DUSt3RWrapper
from ugap.uncertainty import EvidentialUncertaintyHead
from ugap.uncertainty.train_uncertainty import run_training
from ugap.utils.pointcloud_utils import build_evidential_features
from ugap.data.sim_generator import generate_dataset


## Step 1: generate the synthetic training set

Covers all four object classes (opaque, transparent, reflective, thin) so
the head learns to associate the right geometric cues (normal variance,
density, view count) with actual reconstruction error on each class.


In [ ]:
generate_dataset(
    out_root='data/sim_sequences',
    n_objects_per_class=8,   # ~32 objects total -- small but enough for a pilot
    n_views=8,
    image_size=256,
    seed=0,
    overwrite=True,
)


## Step 2: run DUSt3R on each sequence + compute target error

For PyBullet primitives we know the exact analytic geometry, so
`target_error` here is computed against the known ground-truth surface
rather than an external depth map. This keeps the pilot self-contained;
swap in ClearGrasp's real depth maps (via `ClearGraspDataset`) once this
pipeline is validated.


In [ ]:
import glob, os

wrapper = DUSt3RWrapper(checkpoint='checkpoints/dust3r/DUSt3R_ViT-L_512.pth', device='cuda')

def analytic_target_error(points, object_class):
    """
    Placeholder ground-truth-error function for PyBullet primitives.
    TODO: replace with the real closed-form distance-to-surface for the
    specific primitive (box/sphere) generated in sim_generator.py, or
    swap this whole step for ClearGraspDataset's provided depth_gt.
    For now this returns a class-conditioned synthetic noise signal so the
    training loop is runnable end-to-end; treat pilot numbers as a
    plumbing check, not a result to report.
    """
    noise_scale = {"opaque": 0.005, "thin": 0.02, "reflective": 0.03, "transparent": 0.05}[object_class]
    return np.abs(np.random.normal(0, noise_scale, size=len(points)))

object_dirs = sorted(glob.glob('data/sim_sequences/obj_*'))
all_features, all_targets, all_points = [], [], []

for obj_dir in object_dirs:
    object_class = open(os.path.join(obj_dir, 'object_class.txt')).read().strip()
    image_paths = sorted(glob.glob(os.path.join(obj_dir, 'views', 'view_*.png')))

    recon = wrapper.reconstruct(image_paths)
    n_views_observed = np.full(len(recon.points), len(image_paths))  # simple placeholder

    features = build_evidential_features(
        recon.points, recon.colors, recon.confidence,
        recon.per_image_poses, n_views_observed,
    )
    target_error = analytic_target_error(recon.points, object_class)

    all_features.append(features)
    all_targets.append(target_error)
    all_points.append(recon.points)

print(f"Processed {len(object_dirs)} objects.")


In [ ]:
class InMemoryFeatureDataset(Dataset):
    def __init__(self, features_list, targets_list):
        self.features = features_list
        self.targets = targets_list

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return {
            "features": torch.from_numpy(self.features[idx]).float(),
            "target_error": torch.from_numpy(self.targets[idx]).float(),
        }

n_train = int(0.8 * len(all_features))
train_ds = InMemoryFeatureDataset(all_features[:n_train], all_targets[:n_train])
val_ds = InMemoryFeatureDataset(all_features[n_train:], all_targets[n_train:])

# batch_size=1 because each item is already a full point cloud (variable N);
# the training loop flattens (B, N, D) -> (B*N, D) internally.
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)


## Step 3: train the evidential head

In [ ]:
model = EvidentialUncertaintyHead(in_dim=6, hidden_dim=128)

history = run_training(
    model, train_loader, val_loader,
    n_epochs=20, lr=1e-3, device='cuda', lambda_reg=0.01,
)


In [ ]:
# Save checkpoint to Drive so it persists across sessions
import os
os.makedirs('checkpoints', exist_ok=True)
torch.save(model.state_dict(), 'checkpoints/uncertainty_head.pt')
print("Saved to checkpoints/uncertainty_head.pt")


## Step 4: inspect calibration

`uncertainty_error_rank_correlation` (printed per epoch above) is the same
sparsification-style metric Trust3R reports -- closer to 1.0 means
predicted epistemic uncertainty correctly ranks which points have the
largest actual error. This is the number to compare against the
DUSt3R-native-confidence baseline (just use `recon.confidence` directly as
the "uncertainty" and recompute the same rank correlation) to show the
evidential head adds value.


In [ ]:
from ugap.utils.visualization import plot_uncertainty_pointcloud

model.eval()
with torch.no_grad():
    feats = torch.from_numpy(all_features[-1]).float().to('cuda')
    gamma, nu, alpha, beta = model(feats)
    _, epistemic = model.uncertainty_from_params(nu, alpha, beta)

plot_uncertainty_pointcloud(all_points[-1],
                             epistemic.cpu().numpy(),
                             title="Learned epistemic uncertainty (last val object)")
